In [0]:
CATALOG = "workspace"
SCHEMA = "mortgage_agent"

spark.sql(f"USE CATALOG {CATALOG}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
print(f"Schema ready: {CATALOG}.{SCHEMA}")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.users (
              id STRING NOT NULL,
              created_at TIMESTAMP,
              updated_at TIMESTAMP,
              name STRING,
              email STRING,
              income DOUBLE,
              dp_amount DOUBLE,
              dp_percent DOUBLE,
              location STRING,
              province STRING,
              is_first_time BOOLEAN,
              purchase_type STRING,
              target_price DOUBLE,
              liabilities STRING,
              preferences STRING
          )
          USING DELTA
          """)
print("users table ready")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.sessions (
              id STRING NOT NULL,
              user_id STRING NOT NULL,
              created_at TIMESTAMP,
              updated_at TIMESTAMP,
              progress_json STRING,
              mode STRING,
              is_active BOOLEAN
          )
          USING DELTA
          """)

print("sessions table ready")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.messages (
              id STRING NOT NULL,
              session_id STRING NOT NULL,
              role STRING NOT NULL,
              content STRING NOT NULL,
              created_at TIMESTAMP,
              tokens_used INT,
              metadata STRING
          )
          USING DELTA
          """)

print("messages table ready")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.knowledge_topics (
              id STRING NOT NULL,
              category STRING NOT NULL,
              title STRING NOT NULL,
              slug STRING NOT NULL,
              content STRING NOT NULL,
              tags ARRAY<STRING>,
              difficulty STRING,
              sort_order INT,
              is_active BOOLEAN  
          )
          USING DELTA
          """)
print("knowledge_topics table ready")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.user_progress (
              id STRING NOT NULL,
              user_id STRING NOT NULL,
              session_id STRING,
              topic_id STRING NOT NULL,
              covered_at TIMESTAMP,
              depth STRING 
          )
          USING DELTA
          """)
print("user_progress table ready")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.agreements (
              id STRING NOT NULL,
              user_id STRING NOT NULL,
              session_id STRING,
              created_at TIMESTAMP,
              filename STRING NOT NULL,
              file_size_bytes INT,
              builder_name STRING,
              project_name STRING,
              clauses_json STRING,
              risk_summary STRING,
              status STRING
          )
          USING DELTA
          """)
print("agreements table ready")

In [0]:
tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()

print(f"Tables in {CATALOG}.{SCHEMA}:")
for t in tables:
    name = t["tableName"]
    count = spark.sql(f"SELECT count(*) as cnt FROM {CATALOG}.{SCHEMA}.{name}").collect()[0]["cnt"]
    print(f"  {name}: {count} rows")

In [0]:
import uuid

def tid():
    return str(uuid.uuid4())[:8]

KNOWLEDGE_TOPICS = [
    (tid(), "regulatory", "Mortgage Stress Test (B-20)", "stress-test-b20",
     "All federally regulated lenders must qualify borrowers at the higher of: (a) the mortgage contract rate + 2%, or (b) the Bank of Canada's benchmark qualifying rate. This means even if you get a 4% rate, you must prove you can afford payments at 6%. Applies to new mortgages, renewals with a new lender, and refinances. OSFI Guideline B-20 governs this.",
     ["stress test", "b-20", "qualification", "qualifying rate", "osfi"], "beginner", 1),
    
    (tid(), "regulatory", "OSFI B-21 (Mortgage Insurance)", "osfi-b21", 
     "Guideline B-21 sets standards for mortgage insurers (CMHC, Sagen, Canada Guaranty). Requires insurers to independently verify borrower eligibility and apply stress tests. Max amortization 25 years for insured. Purchase price must be under $1.5M. Minimum DP: 5% on first $500K, 10% on $500K-$1.5M.",
     ["b-21", "mortgage insurance", "cmhc", "osfi", "insurer"], "intermediate", 2),

    (tid(), "regulatory", "FCAC Consumer Protection", "fcac-consumer", 
     "FCAC oversees consumer protection in banking. Lenders must provide mortgage disclosure within 2 business days. Borrowers can prepay up to 20% of original principal per year without penalty. FCAC requires clear disclosure of IRD penalty calculations.", 
     ["fcac", "consumer protection", "disclosure", "prepayment"], "intermediate", 3), 
    
    (tid(), "down_payment", "Down Payment Requirements", "down-payment-requirements", 
     "Canadian minimums: 5% on first $500K, 10% on $500K-$1.5M, 20% on $1.5M+. Example: $700K home = $25K + $20K = $45K minimum. Under 20% triggers mandatory CMHC insurance. Gift funds allowed with gift letter from immediate family.", 
     ["down payment", "minimum", "5 percent", "10 percent", "20 percent", "gift"], "beginner", 10),
    
    (tid(), "down_payment", "CMHC Mortgage Insurance", "cmhc-insurance", 
     "When DP < 20%, insurance is mandatory. Providers: CMHC, Sagen, Canada Guaranty. Premiums: 5-9.99% DP = 4.00%, 10-14.99% = 3.10%, 15-19.99% = 2.80%. Premium added to mortgage balance. Max insurable amortization: 25 years. Max price: $1.5M.", 
     ["cmhc", "mortgage insurance", "sagen", "canada guaranty", "premium", "insured"], "beginner", 11),
    
    (tid(), "rates", "Fixed Rate Mortgages", "fixed-rate", 
     "Rate locked for the entire term (1-5 years). Tied to Government of Canada bond yields, not BoC rate. Penalty for breaking: greater of 3 months interest or IRD. IRD penalties can be $15K-$30K+. Best for buyers wanting payment certainty.", 
     ["fixed rate", "bond yield", "ird", "penalty", "predictable"], "beginner", 20),
    
    (tid(), "rates", "Variable Rate Mortgages", "variable-rate", 
     "Fluctuates with lender's prime rate (tracks BoC overnight rate). Quoted as prime minus X%. Two types: adjustable payment and fixed payment. Trigger rate risk with fixed-payment variable. Penalty: typically 3 months interest only. Best for buyers tolerating uncertainty.", 
     ["variable rate", "prime rate", "adjustable", "trigger rate", "bank of canada"], "beginner", 21),

    (tid(), "rates", "Rate Holds and Pre-Approval", "rate-hold", 
     "Rate hold locks a quoted rate for 90-120 days. If rates rise, you keep the lower rate. If rates fall, most lenders give lower rate at closing. Pre-approval is NOT a guarantee. Conditions still apply: appraisal, title search, income verification.", 
     ["rate hold", "pre-approval", "lock", "90 days", "conditional"], "beginner", 22),

    (tid(), "affordability", "GDS and TDS Ratios", "gds-tds-ratios", 
     "GDS: housing costs / gross income. Max ~39% insured, 35% uninsured. TDS: GDS + all debt payments / gross income. Max ~44% insured, 42% uninsured. Calculated at stress test rate, not actual rate. Exceeding limits = disqualification.", 
     ["gds", "tds", "debt ratio", "affordability", "qualification"], "intermediate", 30),
    
    (tid(), "affordability", "Loan-to-Value (LTV)", "ltv", 
     "LTV = mortgage / property value. Above 80% LTV requires CMHC insurance. Max LTV: 95% purchase, 80% refinance, 65% HELOC. Combined mortgage + HELOC cannot exceed 80%. Higher LTV may mean higher rates on uninsured mortgages.", 
     ["ltv", "loan to value", "80 percent", "95 percent", "equity"], "beginner", 31),
    
    (tid(), "costs", "Closing Costs Breakdown", "closing-costs", 
     "Budget 1.5-4% of purchase price. Land transfer tax (Ontario tiers: 0.5%-2.5%, Toronto adds municipal LTT). Legal fees $1.5-2.5K. Title insurance $300-500. Home inspection $400-600. Appraisal $300-500. First-time Ontario buyers get LTT rebate up to $4,000.", 
     ["closing costs", "land transfer tax", "legal fees", "ltt", "toronto"], "beginner", 40),
    
    (tid(), "costs", "IRD Penalty Calculation", "ird-penalty", 
     "IRD = (your rate - current comparable rate) x balance x remaining term. Big 5 banks use posted rates (inflated), resulting in huge penalties. Monolines use actual discount rates (lower penalties). Example: 1% diff x $400K x 3yr = $12K. Always ask for written estimate.", 
     ["ird", "penalty", "interest rate differential", "break", "posted rate"], "advanced", 41),
    
    (tid(), "pre_construction", "Pre-Construction Purchase Risks", "pre-construction-risks", 
     "Closing delays up to 2 years via interim occupancy. During interim occupancy you pay fees but don't own the unit. Assignment restrictions and fees common. Cancellation rights under Tarion. Price escalation clauses. Parking/locker costs may not be finalized.", 
     ["pre-construction", "builder", "interim occupancy", "assignment", "tarion"], "intermediate", 50),
    
    (tid(), "pre_construction", "Tarion Warranty Coverage", "tarion-warranty", 
     "Ontario new home warranty: deposit protection ($40K freehold, $20K condo), 1-year (workmanship), 2-year (water, plumbing, heating, electrical), 7-year (major structural). Must complete PDI and 30-Day Form. Check builder's claim history.", 
     ["tarion", "warranty", "new home", "ontario", "pdi"], "intermediate", 51),
    
    (tid(), "pre_construction", "HCRA Builder Licensing", "hcra-builder", 
     "HCRA licenses builders and vendors in Ontario. Verify license status on HCRA Ontario Builder Directory before signing. Check: active license, conditions, complaint history, suspensions. Unlicensed builder = no legal protection for your deposit.", 
     ["hcra", "builder", "license", "ontario", "directory"], "intermediate", 52),
    
    (tid(), "products", "Standard Closed Mortgage", "standard-closed", 
     "Most common type. Locked into term with penalties for early break. Fixed or variable. Terms 1-5 years. Amortization up to 25yr insured, 30yr uninsured. Collateral vs standard charge affects portability at renewal.", 
     ["closed mortgage", "standard", "term", "amortization", "collateral charge"], "beginner", 60),
    
    (tid(), "products", "Open Mortgage", "open-mortgage", 
     "Can be paid off anytime without penalty. Rate is 1-2% higher than closed. Best for buyers expecting to sell soon or receive lump sum. Typically short terms (6 months - 1 year). Consider closed mortgage with good prepayment privileges instead.", 
     ["open mortgage", "no penalty", "flexible", "higher rate"], "beginner", 61),
    
    (tid(), "products", "HELOC (Home Equity Line of Credit)", "heloc", 
     "Borrow against home equity up to 65% LTV. Revolving credit. Rate: prime + 0.5% to 1.0%. Interest-only payments. Combined with mortgage cannot exceed 80% LTV. Readvanceable mortgage increases HELOC as principal paid down. Requires 20% equity.", 
     ["heloc", "home equity", "line of credit", "revolving", "readvanceable"], "intermediate", 62),
    
    (tid(), "products", "Readvanceable Mortgage + Smith Manoeuvre", "readvanceable-smith", 
     "Combines mortgage and HELOC under one collateral charge. As mortgage principal paid, HELOC limit increases. Smith Manoeuvre: borrow from HELOC to invest, making interest tax-deductible. Converts non-deductible to deductible debt. Requires discipline.", 
     ["readvanceable", "smith manoeuvre", "tax deductible", "heloc", "collateral charge"], "advanced", 63),
    
    (tid(), "programs", "First Home Savings Account (FHSA)", "fhsa", 
     "FHSA: $8K/year, $40K lifetime. Contributions tax-deductible (like RRSP). Withdrawals for home purchase tax-free (like TFSA). Must be first-time buyer. Can combine with HBP. Account open 1+ year before withdrawal. Unused room carries forward.", 
     ["fhsa", "first home savings", "tax free", "tax deductible", "first time buyer"], "beginner", 70),
    
    (tid(), "programs", "Home Buyers' Plan (HBP)", "hbp", 
     "Withdraw up to $60K from RRSP tax-free ($120K per couple). Repay over 15 years starting year 2. Missed repayment = taxable income. RRSP funds must be deposited 90+ days before withdrawal. Can combine with FHSA.", 
     ["hbp", "home buyers plan", "rrsp", "withdrawal", "first time buyer"], "beginner", 71),
    
    (tid(), "process", "Mortgage Approval Process", "approval-process", 
     "Stages: 1) Pre-qualification (informal). 2) Pre-approval (credit check + rate hold 90-120 days). 3) Firm offer with signed purchase agreement. 4) Underwriting: verify income, employment, appraisal, title. 5) Conditions issued. 6) Fulfillment. 7) Closing. Timeline: 2-4 weeks offer to conditions, 30-60 days to close.", 
     ["approval", "process", "underwriting", "conditions", "closing", "timeline"], "beginner", 80),
]

print(f"Prepared {len(KNOWLEDGE_TOPICS)} topics")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType, BooleanType
from pyspark.sql import Row

schema = StructType([
    StructField("id", StringType()),
    StructField("category", StringType()),
    StructField("title", StringType()),
    StructField("slug", StringType()),
    StructField("content", StringType()),
    StructField("tags", ArrayType(StringType())),
    StructField("difficulty", StringType()),
    StructField("sort_order", IntegerType()),
])

rows = [Row(id=t[0], category=t[1], title=t[2], slug=t[3], content=t[4], tags=t[5], difficulty=t[6], sort_order=t[7]) for t in KNOWLEDGE_TOPICS]

df =spark.createDataFrame(rows, schema)
df.createOrReplaceTempView("new_topics")

print(f"total number of rows: {df.count()}")

spark.sql(f"""
          MERGE INTO {CATALOG}.{SCHEMA}.knowledge_topics AS target
          USING new_topics AS source
          ON target.slug = source.slug
          WHEN MATCHED THEN UPDATE SET
            target.category = source.category,
            target.title = source.title,
            target.content = source.content,
            target.tags = source.tags,
            target.difficulty = source.difficulty,
            target.sort_order = source.sort_order,
            target.is_active = true
          WHEN NOT MATCHED THEN INSERT (
              id, category, title, slug, content, tags, difficulty, sort_order, is_active
            ) 
          VALUES (
              source.id, source.category, source.title, source.slug, source.content, source.tags, source.difficulty, source.sort_order, true
          )
          """)

count = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.knowledge_topics").collect()[0][0]
print(f"total number of rows after merge: {count}")

In [0]:
display(
    spark.sql(f"""
              SELECT category, count(*) as topic_count FROM {CATALOG}.{SCHEMA}.knowledge_topics
              WHERE is_active = true
              GROUP BY category
              ORDER BY category
              """)
)